# 🚀 Treinamento LoRA SDXL — Persona Scarlett (20 Fotos Consistentes)
**Treinamento autônomo de consistência facial e trajes (biquíni, moda praia, corpo inteiro, estúdio).**

Execute cada célula em sequência (ou `Runtime -> Run all`).

In [ ]:
# @title 1. Verificar GPU NVIDIA (Tesla T4 / A100 / L4)
import torch
assert torch.cuda.is_available(), 'ERRO: Nenhuma GPU detectada. Vá em Runtime -> Change runtime type -> Escolha T4 GPU.'
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ GPU Ativa: {gpu} ({vram:.2f} GB VRAM)')
!nvidia-smi

In [ ]:
# @title 2. Instalar Dependências ML
!pip install --quiet diffusers transformers accelerate peft bitsandbytes torchvision
print('✅ Dependências instaladas com sucesso!')

In [ ]:
# @title 3. Baixar Dataset de 20 Fotos Estruturadas (Mesmo Rosto + Captions)
import os, urllib.request, zipfile

dataset_url = 'https://github.com/HawkEyesz/persona-lora-datasets/releases/download/v1.0.0/persona_lora_dataset_20.zip'
zip_dest = '/tmp/dataset_20.zip'
dataset_dir = '/content/dataset_scarlett_20'
os.makedirs(dataset_dir, exist_ok=True)

print('Baixando dataset estruturado do release oficial...')
urllib.request.urlretrieve(dataset_url, zip_dest)

with zipfile.ZipFile(zip_dest, 'r') as z:
    z.extractall(dataset_dir)

files = sorted(os.listdir(dataset_dir))
print(f'✅ {len(files)} arquivos extraídos com sucesso em {dataset_dir}:')
for f in files[:6]:
    print(f'  - {f}')

In [ ]:
# @title 4. Executar Treinamento LoRA SDXL (500 Steps / Rank 32 / 8-bit AdamW)
import os, math, gc, time, json
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from diffusers import UNet2DConditionModel, AutoencoderKL, DDPMScheduler
from transformers import CLIPTextModel, CLIPTextModelWithProjection, CLIPTokenizer
from peft import LoraConfig, get_peft_model
import bitsandbytes as bnb

device = torch.device('cuda:0')
model_id = 'stabilityai/stable-diffusion-xl-base-1.0'

class PersonaDataset(Dataset):
    def __init__(self, folder, size=1024):
        self.folder = folder
        self.images = []
        self.captions = []
        self.transform = transforms.Compose([
            transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
        for f in sorted(os.listdir(folder)):
            if f.endswith(('.png', '.jpg')):
                self.images.append(os.path.join(folder, f))
                txt_f = os.path.join(folder, os.path.splitext(f)[0] + '.txt')
                if os.path.exists(txt_f):
                    with open(txt_f, 'r') as c:
                        self.captions.append(c.read().strip())
                else:
                    self.captions.append('photo of sclt_persona woman, youthful 24yo, porcelain skin without freckles')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        return {'pixel_values': self.transform(img), 'caption': self.captions[idx]}

dataset = PersonaDataset(dataset_dir, size=1024)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
print(f'Dataset carregado com {len(dataset)} amostras.')

print('Carregando backbone SDXL...')
tokenizer_one = CLIPTokenizer.from_pretrained(model_id, subfolder='tokenizer')
tokenizer_two = CLIPTokenizer.from_pretrained(model_id, subfolder='tokenizer_2')
text_encoder_one = CLIPTextModel.from_pretrained(model_id, subfolder='text_encoder', torch_dtype=torch.float16).to(device)
text_encoder_two = CLIPTextModelWithProjection.from_pretrained(model_id, subfolder='text_encoder_2', torch_dtype=torch.float16).to(device)
vae = AutoencoderKL.from_pretrained(model_id, subfolder='vae', torch_dtype=torch.float16).to(device)
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder='unet', torch_dtype=torch.float16).to(device)
noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder='scheduler')

vae.requires_grad_(False)
text_encoder_one.requires_grad_(False)
text_encoder_two.requires_grad_(False)
unet.requires_grad_(False)
unet.enable_gradient_checkpointing()

# LoRA config
lora_config = LoraConfig(
    r=32, lora_alpha=64, init_lora_weights='gaussian',
    target_modules=['to_k', 'to_q', 'to_v', 'to_out.0']
)
unet.add_adapter(lora_config)

optimizer = bnb.optim.AdamW8bit(unet.parameters(), lr=1e-4)
num_epochs = 25
total_steps = len(dataloader) * num_epochs
global_step = 0
unet.train()

print(f'Iniciando Treinamento LoRA ({total_steps} passos)...')
pbar = tqdm(range(total_steps))
for epoch in range(num_epochs):
    for step, batch in enumerate(dataloader):
        pixel_values = batch['pixel_values'].to(device, dtype=torch.float16)
        captions = batch['caption']
        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample() * vae.config.scaling_factor
            t1 = tokenizer_one(captions, padding='max_length', max_length=77, truncation=True, return_tensors='pt').to(device)
            t2 = tokenizer_two(captions, padding='max_length', max_length=77, truncation=True, return_tensors='pt').to(device)
            prompt_embeds1 = text_encoder_one(t1.input_ids)[0]
            enc_out2 = text_encoder_two(t2.input_ids, output_hidden_states=True)
            prompt_embeds2 = enc_out2.hidden_states[-2]
            pooled_prompt_embeds = enc_out2[0]
            prompt_embeds = torch.concat([prompt_embeds1, prompt_embeds2], dim=-1)
            add_time_ids = torch.tensor([[1024, 1024, 0, 0, 1024, 1024]], dtype=torch.float16, device=device)
            added_cond_kwargs = {'text_embeds': pooled_prompt_embeds, 'time_ids': add_time_ids}
            noise = torch.randn_like(latents)
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=latents.device).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        
        model_pred = unet(noisy_latents, timesteps, prompt_embeds, added_cond_kwargs=added_cond_kwargs).sample
        loss = F.mse_loss(model_pred.float(), noise.float(), reduction='mean')
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        global_step += 1
        pbar.update(1)
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'step': global_step})

out_dir = '/content/persona_scarlett_sdxl_lora'
os.makedirs(out_dir, exist_ok=True)
unet.save_attn_procs(out_dir)
print(f'✅ Treinamento LoRA concluído com sucesso! Pesos salvos em {out_dir}.')

In [ ]:
# @title 5. Teste de Inferência com o Novo LoRA (Gerar Imagem de Validação)
from diffusers import StableDiffusionXLPipeline
import torch

print('Carregando pipeline de teste com o LoRA montado...')
pipe = StableDiffusionXLPipeline.from_pretrained(
    model_id, torch_dtype=torch.float16, variant='fp16'
).to(device)
pipe.load_lora_weights('/content/persona_scarlett_sdxl_lora')

prompt = 'photo of sclt_persona woman, youthful 24yo, wearing luxury red bikini at luxury resort pool, flawless smooth pale porcelain skin without freckles, beautiful toned body, wet skin glistening, golden sunlight, 8k uhd dslr'
negative = 'freckles, blemishes, spots, wrinkles, older, mature, 30s, ugly, distorted, deformed, bad anatomy, bad hands'

image = pipe(prompt=prompt, negative_prompt=negative, num_inference_steps=30, guidance_scale=6.5).images[0]
image.save('/content/validacao_lora_scarlett.png')
image